# Práctica Guiada: Preparación de Datos y Exploración

¡Bienvenido/a a la plantilla **Plug and Play**! Este notebook está listo para ejecutarse celda por celda.
Aprenderás y pondrás en práctica los cuatro pilares fundamentales del preprocesamiento de datos:
1. **Gestión de Duplicados**
2. **Tratamiento de Valores Faltantes (Missing Values)**
3. **Estandarización de Variables** (`StandardScaler`)
4. **Detección de Outliers y Exploración Visual**


## Módulo 1: Importación y Carga de Datos

En esta sección importamos las librerías científicas de Python (`pandas`, `numpy`, `matplotlib`, `seaborn`) y cargamos nuestro conjunto de datos.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 1. Importar librerías principales
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración estética para gráficos
%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted")

# 2. Cargar el dataset

import pandas as pd
data = pd.read_csv("/content/drive/MyDrive/Clases Diplomado/películas.csv")


# 3. Vista previa general de la estructura
display(data.head(5))
print(f"Dimensiones originales: {data.shape[0]} filas y {data.shape[1]} columnas.")

## Módulo 2: Detección y Limpieza de Duplicados

Los registros repetidos pueden sesgar los modelos de Machine Learning. Verificaremos cuántos existen y los depuraremos.

In [ ]:
# Asignar el dataset cargado a la variable 'df'
df = data

# Identificar filas duplicadas
duplicados = df[df.duplicated()]
print(f"Cantidad de filas duplicadas encontradas: {len(duplicados)}")

# Eliminar duplicados de manera permanente
df.drop_duplicates(inplace=True)
print(f"Dimensiones después de eliminar duplicados: {df.shape}")

## Módulo 3: Tratamiento de Valores Faltantes (Missing Values)

En la vida real, los datos suelen venir incompletos. Vamos a simular nulos temporalmente para aprender a gestionarlos usando dos enfoques:
- **Eliminación (`dropna`)**
- **Imputación/Sustitución (`fillna`)**

In [ ]:
# Creamos una copia para experimentar con nulos educativos
df_missing = df.copy()

np.random.seed(42)
df_missing.iloc[::4, 2] = np.nan  # Introducimos nulos en 'cuota'
df_missing.iloc[::6, 5] = np.nan  # Introducimos nulos en 'PDI'

# 1. Inspeccionar valores faltantes por columna
print("Conteo de valores nulos (NaN) introducidos:")
print(df_missing.isna().sum())

In [ ]:
# 2. Opción A: Eliminar filas con nulos (útil si hay pocos faltantes)
df_clean_dropna = df_missing.dropna()
print(f"Filas tras dropna(): {len(df_clean_dropna)}")

In [ ]:
# 3. Opción B: Rellenar nulos (Imputación por la media o valor constante)
df_clean_fillna = df_missing.copy()
df_clean_fillna['vote_average'] = df_clean_fillna['vote_average'].fillna(df_clean_fillna['vote_average'].mean())
df_clean_fillna['revenue'] = df_clean_fillna['revenue'].fillna(df_clean_fillna['revenue'].median())

print("Nulos después de aplicar fillna() en las columnas reales:")
print(df_clean_fillna.isna().sum())

## Módulo 4: Estandarización de Datos (`StandardScaler`)

Estandarizar transforma las variables para que tengan **media = 0** y **desviación estándar = 1**, evitando que variables con escalas grandes dominen sobre las demás.

In [ ]:
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

# Seleccionamos solo las columnas numéricas de los datos limpios para escalar
df_numerico = df.dropna().select_dtypes(include=[np.number])

# Aplicar StandardScaler
scaler = StandardScaler()
scaled_array = scaler.fit_transform(df_numerico)
df_scaled = pd.DataFrame(scaled_array, columns=df_numerico.columns)

In [ ]:
# Comprobación matemática
print("Medias tras escalar (deben ser prácticamente 0):")
print(df_scaled.mean().round(4))

print("\nDesviaciones estándar tras escalar (deben ser 1):")
print(df_scaled.std().round(4))

In [ ]:
# Visualizar impacto con Boxplots
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df_numerico.boxplot(ax=axes[0])
axes[0].set_title('Datos Originales (Escalas diversas)')
axes[0].tick_params(axis='x', rotation=45)

df_scaled.boxplot(ax=axes[1])
axes[1].set_title('Datos Estandarizados (StandardScaler)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## Módulo 5: Detección y Tratamiento de Outliers

Analizaremos los valores atípicos mediante gráficos estadísticos y reglas de filtrado basadas en la desviación estándar.

In [ ]:
# Analizar la variable 'revenue'
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
sns.histplot(df_numerico['revenue'], kde=True, color='teal')
plt.title('Distribución de Revenue')

plt.subplot(1, 2, 2)
sns.boxplot(x=df_numerico['revenue'], color='coral')
plt.title('Boxplot de Revenue')
plt.tight_layout()
plt.show()

In [ ]:
# Filtrado de outliers usando el criterio de 3 desviaciones estándar (Z-Score)
import numpy as np

columna_obj = 'revenue'
media = df_numerico[columna_obj].mean()
desv = df_numerico[columna_obj].std()

df_sin_outliers = df_numerico[
    np.abs(df_numerico[columna_obj] - media) <= 3 * desv
]
print(f"Registros originales: {len(df_numerico)} | Tras filtrar outliers: {len(df_sin_outliers)}")

## Módulo 6: Exploración Analítica (Univariada y Bivariada)

Finalizamos resumiendo métricas estadísticas y segmentando la información según el perfil de riesgo crediticio.

In [ ]:
# 1. Estadísticas descriptivas generales
print("Resumen estadístico de las variables numéricas:")
display(df_numerico.describe())

# 2. Relación bivariada visual rápida entre variables reales de películas
variables_clave = ['vote_average', 'revenue', 'runtime', 'budget']
sns.pairplot(df_numerico[variables_clave], plot_kws={'alpha': 0.5})
plt.suptitle('Relaciones entre variables del Dataset de Películas', y=1.02)
plt.show()